In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
BASE_PATH = "/content/drive/MyDrive/quickpay-fintech-case-study"

In [3]:
import pandas as pd

transactions = pd.read_csv(
    f"{BASE_PATH}/01_data/raw/transactions_raw.csv"
)

In [4]:
!pip install pandas numpy matplotlib seaborn


In [5]:
transactions.head()

,transaction_id,transaction_date,merchant_name,raw_amount,currency,status,risk_score,gateway_region,user_id,payment_method
0,T001,2026-03-01,alpha mart,420000,INR,captured,score:62,APAC,U001,UPI
1,T002,2026-03-01,ALPHA MART,210000,INR,Captured,55,NaN,U002,Card
2,T003,2026-03-01,BETA STORES,510000,INR,CAPTURED,71,APAC,U003,NetBanking
3,T004,2026-03-02,Beta Stores,160000,INR,failed e05 timeout,68,apac,U004,Card
4,T005,2026-03-02,Alpha Mart,390000,INR,CAPTURED,58,NaN,U001,UPI


In [6]:
import sqlite3
conn = sqlite3.connect("quickpay.db")

In [8]:
import pandas as pd

df = pd.read_csv('cleaned_transactions.csv')

df.head()

,transaction_id,transaction_id.1,transaction_date,transaction_date_clean,merchant_name,merchant_name_clean,raw_amount,currency,status,status_clean,...,user_id,payment_method,payment_method_clean,amount_usd,merchant_id,account_manager,merchant_category,default_region,high_value_flag,high_risk_flag
0,T001,T001,2026-03-01,3/1/2026,alpha mart,ALPHA MART,420000.0,INR,captured,CAPTURED,...,U001,UPI,UPI,4998.0,M001,Aisha Khan,Grocery,APAC,0.0,0.0
1,T002,T002,2026-03-01,3/1/2026,ALPHA MART,ALPHA MART,210000.0,INR,Captured,CAPTURED,...,U002,Card,CARD,2499.0,M001,Aisha Khan,Grocery,APAC,0.0,0.0
2,T003,T003,2026-03-01,3/1/2026,BETA STORES,BETA STORES,510000.0,INR,CAPTURED,CAPTURED,...,U003,NetBanking,NETBANKING,6069.0,M002,Rohan Mehta,Electronics,APAC,1.0,0.0
3,T004,T004,2026-03-02,3/2/2026,Beta Stores,BETA STORES,160000.0,INR,failed e05 timeout,FAILED,...,U004,Card,CARD,1904.0,M002,Rohan Mehta,Electronics,APAC,0.0,0.0
4,T005,T005,2026-03-02,3/2/2026,Alpha Mart,ALPHA MART,390000.0,INR,CAPTURED,CAPTURED,...,U001,UPI,UPI,4641.0,M001,Aisha Khan,Grocery,APAC,0.0,0.0


In [9]:
df.to_sql(
    'transactions',
    conn,
    if_exists='replace',
    index=False
)

32

In [10]:
query_q1 = """

SELECT
    status_clean,
    COUNT(*) AS transaction_count
FROM transactions
GROUP BY status_clean

"""

result_q1 = pd.read_sql(query_q1, conn)

result_q1

,status_clean,transaction_count
0,None,2
1,CAPTURED,19
2,CHARGEBACK,4
3,FAILED,7


In [12]:
#question 2
query_q2 = """

SELECT
    merchant_name_clean,
    SUM(amount_usd) AS total_captured_gmv
FROM transactions
WHERE status_clean = 'CAPTURED'
GROUP BY merchant_name_clean
ORDER BY total_captured_gmv DESC

"""

result_q2 = pd.read_sql(query_q2, conn)

result_q2

,merchant_name_clean,total_captured_gmv
0,BETA STORES,33141.5
1,ALPHA MART,29928.5
2,DELTA TRAVELS,10300.0
3,CITY PHARMA,8640.0


In [13]:
query_q3 = """

SELECT
    merchant_name_clean,
    SUM(amount_usd) AS total_captured_gmv
FROM transactions
WHERE status_clean = 'CAPTURED'
GROUP BY merchant_name_clean
ORDER BY total_captured_gmv DESC
LIMIT 10

"""

result_q3 = pd.read_sql(query_q3, conn)

result_q3

,merchant_name_clean,total_captured_gmv
0,BETA STORES,33141.5
1,ALPHA MART,29928.5
2,DELTA TRAVELS,10300.0
3,CITY PHARMA,8640.0


In [20]:
query_q4 = """

SELECT
    transaction_date_clean,
    SUM(amount_usd) AS daily_gmv,
    COUNT(*) AS successful_transaction_count
FROM transactions
WHERE status_clean = 'CAPTURED'
GROUP BY transaction_date_clean
ORDER BY transaction_date_clean

"""

result_q4 = pd.read_sql(query_q4, conn)

result_q4

,transaction_date_clean,daily_gmv,successful_transaction_count
0,3/1/2026,26382.0,5
1,3/2/2026,11013.5,3
2,3/3/2026,15816.5,4
3,3/4/2026,13804.0,4
4,3/5/2026,6188.0,1
5,3/6/2026,8806.0,2


In [19]:
##5
query_q5 = """

SELECT
    merchant_name_clean,

    COUNT(*) AS total_transactions,

    SUM(
        CASE
            WHEN status_clean = 'CHARGEBACK'
            THEN 1
            ELSE 0
        END
    ) AS chargeback_count,

    ROUND(
        (
            SUM(
                CASE
                    WHEN status_clean = 'CHARGEBACK'
                    THEN 1
                    ELSE 0
                END
            ) * 100.0
        ) / COUNT(*),
        2
    ) AS chargeback_ratio

FROM transactions

GROUP BY merchant_name_clean

HAVING chargeback_ratio > 1

ORDER BY chargeback_ratio DESC

"""

result_q5 = pd.read_sql(query_q5, conn)

result_q5

,merchant_name_clean,total_transactions,chargeback_count,chargeback_ratio
0,ECO HOME,2,1,50.00
1,DELTA TRAVELS,4,1,25.00
2,BETA STORES,11,1,9.09
3,ALPHA MART,11,1,9.09


In [22]:
#6
query_q6 = """

SELECT
    gateway_region_clean,

    COUNT(*) AS total_transactions,

    ROUND(
        AVG(risk_score_clean),
        2
    ) AS average_risk_score

FROM transactions

GROUP BY gateway_region_clean

HAVING
    average_risk_score > 50
    AND total_transactions > 20

ORDER BY average_risk_score DESC

"""

result_q6 = pd.read_sql(query_q6, conn)

result_q6

,gateway_region_clean,total_transactions,average_risk_score


In [23]:
result_q6.to_csv(
    'region_breakdown.csv',
    index=False
)

In [24]:
query_q7 = """

SELECT
    user_id,
    transaction_date_clean,

    COUNT(*) AS suspicious_transaction_count

FROM transactions

WHERE
    status_clean IN ('FAILED', 'CHARGEBACK')

GROUP BY
    user_id,
    transaction_date_clean

HAVING
    suspicious_transaction_count >= 3

ORDER BY
    suspicious_transaction_count DESC

"""

result_q7 = pd.read_sql(query_q7, conn)

result_q7

,user_id,transaction_date_clean,suspicious_transaction_count
0,U008,3/5/2026,4


In [25]:
query_payment_method = """

SELECT
    payment_method_clean,

    COUNT(*) AS total_transactions,

    ROUND(
        SUM(amount_usd),
        2
    ) AS total_gmv

FROM transactions

WHERE
    status_clean = 'CAPTURED'

GROUP BY
    payment_method_clean

ORDER BY
    total_gmv DESC

"""

payment_method_breakdown = pd.read_sql(
    query_payment_method,
    conn
)

payment_method_breakdown

,payment_method_clean,total_transactions,total_gmv
0,CARD,8,34123.0
1,UPI,6,28441.0
2,NETBANKING,2,11662.0
3,WALLET,3,7784.0


In [26]:
payment_method_breakdown.to_csv(
    'payment_method_breakdown.csv',
    index=False
)

In [27]:
query_merchant_performance = """

SELECT
    merchant_name_clean,

    COUNT(*) AS total_transactions,

    SUM(
        CASE
            WHEN status_clean = 'CAPTURED'
            THEN 1
            ELSE 0
        END
    ) AS captured_transactions,

    ROUND(
        SUM(amount_usd),
        2
    ) AS total_gmv,

    SUM(
        CASE
            WHEN status_clean = 'CHARGEBACK'
            THEN 1
            ELSE 0
        END
    ) AS chargeback_count,

    ROUND(
        AVG(risk_score_clean),
        2
    ) AS average_risk_score

FROM transactions

GROUP BY
    merchant_name_clean

ORDER BY
    total_gmv DESC

"""

merchant_performance_summary = pd.read_sql(
    query_merchant_performance,
    conn
)

merchant_performance_summary

,merchant_name_clean,total_transactions,captured_transactions,total_gmv,chargeback_count,average_risk_score
0,BETA STORES,11,7,41531.0,1,51.36
1,ALPHA MART,11,8,40698.0,1,28.91
2,DELTA TRAVELS,4,2,14600.0,1,36.50
3,ECO HOME,2,0,10152.0,1,32.50
4,CITY PHARMA,2,2,8640.0,0,0.00
5,None,2,0,NaN,0,NaN


In [28]:
merchant_performance_summary.to_csv(
    'merchant_performance_summary.csv',
    index=False
)

In [29]:
import pandas as pd
import numpy as np
import json

In [30]:
ledger = pd.read_csv('ledger.csv')

gateway = pd.read_csv('gateway.csv')

In [31]:
ledger.head()

,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,850.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet
3,R004,2026-03-02,M003,2100.0,success,Card
4,R005,2026-03-03,M004,7200.0,success,Card


In [32]:
gateway.head()

,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,900.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet
3,R005,2026-03-03,M004,7200.0,failed,Card
4,R006,2026-03-03,M002,950.0,success,UPI


In [33]:
print("Ledger Shape:", ledger.shape)

print("Gateway Shape:", gateway.shape)

Ledger Shape: (10, 6)
Gateway Shape: (9, 6)


In [34]:
print(ledger.isnull().sum())

print()

print(gateway.isnull().sum())

transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64

transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64


In [35]:
print(
    "Ledger Duplicates:",
    ledger.duplicated().sum()
)

print(
    "Gateway Duplicates:",
    gateway.duplicated().sum()
)

Ledger Duplicates: 0
Gateway Duplicates: 0


In [36]:
missing_in_gateway = ledger[
    ~ledger['transaction_id'].isin(
        gateway['transaction_id']
    )
]

missing_in_gateway

,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
3,R004,2026-03-02,M003,2100.0,success,Card
9,R010,2026-03-05,M004,2500.0,success,Wallet


In [37]:
missing_in_gateway.to_csv(
    'missing_in_gateway.csv',
    index=False
)

In [38]:
missing_in_ledger = gateway[
    ~gateway['transaction_id'].isin(
        ledger['transaction_id']
    )
]

missing_in_ledger

,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
8,R011,2026-03-05,M003,1800.0,success,Card


In [39]:
missing_in_ledger.to_csv(
    'missing_in_ledger.csv',
    index=False
)

In [40]:
merged_data = pd.merge(
    ledger,
    gateway,
    on='transaction_id',
    suffixes=('_ledger', '_gateway')
)

merged_data.head()

,transaction_id,transaction_date_ledger,merchant_id_ledger,amount_usd_ledger,status_ledger,payment_method_ledger,transaction_date_gateway,merchant_id_gateway,amount_usd_gateway,status_gateway,payment_method_gateway
0,R001,2026-03-01,M001,1200.0,success,UPI,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,850.0,success,Card,2026-03-01,M002,900.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet,2026-03-02,M001,500.0,success,Wallet
3,R005,2026-03-03,M004,7200.0,success,Card,2026-03-03,M004,7200.0,failed,Card
4,R006,2026-03-03,M002,950.0,success,UPI,2026-03-03,M002,950.0,success,UPI


In [42]:
amount_mismatches = merged_data[

    merged_data['amount_usd_ledger']
    !=
    merged_data['amount_usd_gateway']

]

amount_mismatches

,transaction_id,transaction_date_ledger,merchant_id_ledger,amount_usd_ledger,status_ledger,payment_method_ledger,transaction_date_gateway,merchant_id_gateway,amount_usd_gateway,status_gateway,payment_method_gateway
1,R002,2026-03-01,M002,850.0,success,Card,2026-03-01,M002,900.0,success,Card
6,R008,2026-03-04,M001,640.0,success,Card,2026-03-04,M001,600.0,success,Card


In [43]:
amount_mismatches.to_csv(
    'amount_mismatches.csv',
    index=False
)

In [44]:
status_mismatches = merged_data[

    merged_data['status_ledger']
    !=
    merged_data['status_gateway']

]

status_mismatches

,transaction_id,transaction_date_ledger,merchant_id_ledger,amount_usd_ledger,status_ledger,payment_method_ledger,transaction_date_gateway,merchant_id_gateway,amount_usd_gateway,status_gateway,payment_method_gateway
3,R005,2026-03-03,M004,7200.0,success,Card,2026-03-03,M004,7200.0,failed,Card


In [45]:
status_mismatches.to_csv(
    'status_mismatches.csv',
    index=False
)

In [46]:
reconciliation_report = merged_data.copy()

reconciliation_report['reconciliation_status'] = np.where(

    reconciliation_report['amount_usd_ledger']
    !=
    reconciliation_report['amount_usd_gateway'],

    'AMOUNT_MISMATCH',

    np.where(

        reconciliation_report['status_ledger']
        !=
        reconciliation_report['status_gateway'],

        'STATUS_MISMATCH',

        'MATCHED'
    )
)

reconciliation_report.head()

,transaction_id,transaction_date_ledger,merchant_id_ledger,amount_usd_ledger,status_ledger,payment_method_ledger,transaction_date_gateway,merchant_id_gateway,amount_usd_gateway,status_gateway,payment_method_gateway,reconciliation_status
0,R001,2026-03-01,M001,1200.0,success,UPI,2026-03-01,M001,1200.0,success,UPI,MATCHED
1,R002,2026-03-01,M002,850.0,success,Card,2026-03-01,M002,900.0,success,Card,AMOUNT_MISMATCH
2,R003,2026-03-02,M001,500.0,success,Wallet,2026-03-02,M001,500.0,success,Wallet,MATCHED
3,R005,2026-03-03,M004,7200.0,success,Card,2026-03-03,M004,7200.0,failed,Card,STATUS_MISMATCH
4,R006,2026-03-03,M002,950.0,success,UPI,2026-03-03,M002,950.0,success,UPI,MATCHED


In [47]:
reconciliation_report.to_csv(
    'reconciliation_report.csv',
    index=False
)

In [48]:
summary_metrics = {

    "total_ledger_rows":
        int(len(ledger)),

    "total_gateway_rows":
        int(len(gateway)),

    "missing_in_gateway_count":
        int(len(missing_in_gateway)),

    "missing_in_ledger_count":
        int(len(missing_in_ledger)),

    "amount_mismatch_count":
        int(len(amount_mismatches)),

    "status_mismatch_count":
        int(len(status_mismatches)),

    "reconciliation_issue_count":
        int(
            len(amount_mismatches)
            +
            len(status_mismatches)
            +
            len(missing_in_gateway)
            +
            len(missing_in_ledger)
        ),

    "ledger_total_amount":
        float(
            ledger['amount_usd'].sum()
        ),

    "gateway_total_amount":
        float(
            gateway['amount_usd'].sum()
        ),

    "amount_at_risk":
        float(
            amount_mismatches['amount_usd_ledger'].sum()
        )
}

summary_metrics

{'total_ledger_rows': 10,
 'total_gateway_rows': 9,
 'missing_in_gateway_count': 2,
 'missing_in_ledger_count': 1,
 'amount_mismatch_count': 2,
 'status_mismatch_count': 1,
 'reconciliation_issue_count': 6,
 'ledger_total_amount': 23340.0,
 'gateway_total_amount': 20550.0,
 'amount_at_risk': 1490.0}

In [49]:
with open(
    'summary_metrics.json',
    'w'
) as file:

    json.dump(
        summary_metrics,
        file,
        indent=4
    )

In [ ]:
##from here on 4th part json normalization starts

In [50]:
with open(
    'api_response_sample.json',
    'r'
) as file:

    api_data = json.load(file)

In [51]:
api_data

{'generated_at': '2026-03-07T10:00:00Z',
 'source': 'QuickPay Settlement API',
 'batches': [{'batch_id': 'B001',
   'merchant': {'merchant_id': 'M001',
    'merchant_name': 'Alpha Mart',
    'region': 'APAC'},
   'settlements': [{'settlement_id': 'S001',
     'amount_usd': 1520.5,
     'status': 'settled',
     'processed_at': '2026-03-07T08:10:00Z',
     'bank': {'name': 'Bank A', 'country': 'IN'}},
    {'settlement_id': 'S002',
     'amount_usd': 980.0,
     'status': 'pending',
     'processed_at': '2026-03-07T08:45:00Z',
     'bank': {'name': 'Bank A', 'country': 'IN'}},
    {'settlement_id': 'S003',
     'amount_usd': 640.0,
     'status': 'settled',
     'processed_at': '2026-03-07T09:15:00Z',
     'bank': {'name': 'Bank B', 'country': 'SG'}}]},
  {'batch_id': 'B002',
   'merchant': {'merchant_id': 'M004',
    'merchant_name': 'Delta Travels',
    'region': 'US'},
   'settlements': [{'settlement_id': 'S004',
     'amount_usd': 2100.0,
     'status': 'settled',
     'processed_at'

In [52]:
api_normalized = pd.json_normalize(api_data)

api_normalized.head()

,generated_at,source,batches
0,2026-03-07T10:00:00Z,QuickPay Settlement API,"[{'batch_id': 'B001', 'merchant': {'merchant_i..."


In [53]:
api_normalized.columns = (

    api_normalized.columns

    .str.lower()

    .str.replace('.', '_')

    .str.replace(' ', '_')

)

In [54]:
api_normalized.head()

,generated_at,source,batches
0,2026-03-07T10:00:00Z,QuickPay Settlement API,"[{'batch_id': 'B001', 'merchant': {'merchant_i..."


In [55]:
for column in api_normalized.columns:

    if 'date' in column or 'time' in column:

        api_normalized[column] = pd.to_datetime(
            api_normalized[column],
            errors='coerce'
        )

In [56]:
api_normalized.to_csv(
    'api_normalized.csv',
    index=False
)